# 迷你 Foundry —— 可以逐格跑、逐格改的版本

`mini_foundry.py` 是一口气跑完的。这个 notebook 把它**拆开**：每一层单独摸一遍，
看中间状态，改改参数再跑。

所有代码都从 `mini_foundry.py` **import**，不复制粘贴——所以这个 notebook 和那个
文件永远不会讲两套故事。想看某个东西怎么实现的，打开那个文件对应的段落。

## 两种模式

| | 要什么 | 能看到什么 |
|---|---|---|
| **剧本模式**（默认） | 什么都不要 | 完整的 agent loop：多轮工具调用、审批、证据核对 |
| **API 模式** | 一个 OpenAI 兼容的 endpoint | 真实的 HTTP 往返、真的模型在决定调什么工具 |

在下面第 5 节切换。先用剧本模式走一遍，再打开 API 模式对照着看。

> **对照阅读**：[README.md](README.md) 把这里的每一节映射到 `src/foundry/` 的真实模块。

In [ ]:
import json, shutil, sys
from pathlib import Path

# 让 notebook 无论从哪里启动都能 import 到 mini_foundry
HERE = Path.cwd()
DEMO = HERE if (HERE / "mini_foundry.py").is_file() else HERE / "demo"
sys.path.insert(0, str(DEMO))

import mini_foundry as mf

RUN = DEMO / ".nb"
print("demo 目录:", DEMO)
print("工作目录:", RUN)


def fresh_workspace():
    """每次都重建，这样任何一格都能反复跑，结果一样。"""
    return mf.make_sample_repo(RUN)


def fresh_session(name="session.jsonl"):
    RUN.mkdir(parents=True, exist_ok=True)
    return mf.Session(RUN / name)

---
## 先记住这一句

```
while 模型还在要求调用工具:
    policy 判决 -> 执行 -> 把结果塞回对话 -> 再问一次模型
```

**其余全部代码，都是在给这句话里的某个词加保护。** 下面就是逐个词看。

---
## 1. IR —— 对话长什么样

对应 `core/conversation.py`。

关键：这套结构**不属于任何模型厂商**。OpenAI 的 JSON、Anthropic 的 JSON 都在
backend 那一层翻译成它，所以换模型不影响 loop / policy / tools 一行代码。

In [ ]:
call = mf.ToolCall(id="c1", name="read_file", arguments='{"path": "calc.py"}')
turn = mf.ModelTurn(text="我先看看这个文件。", tool_calls=[call])

print(turn)
print()
print("arguments 是原始字符串，不是 dict:", repr(call.arguments), type(call.arguments))

`arguments` 故意**不提前解析**。模型是会吐出坏 JSON 的——如果在这一层就 parse，
"这个调用的参数坏了" 这件事就没法报告给模型让它自己改了。下一格看这个。

In [ ]:
broken = mf.ToolCall(id="c9", name="read_file", arguments='{"path": ')
tools = mf.Tools(fresh_workspace())

try:
    tools.validate(broken)
except ValueError as exc:
    print("validate 拒绝了它：", exc)
    print()
    print("→ loop 会把这句话作为工具结果还给模型，模型下一轮通常就改对了。")

---
## 2. 工具 —— 模型能做的事

对应 `core/tools/`。每个工具两件事：`validate`（这个调用合法吗）和 `run`（做）。

**`validate` 必须在 policy 之前跑。** 否则一个畸形调用会先弹审批框给用户，
用户批准了才发现参数根本不对。

In [ ]:
workspace = fresh_workspace()
tools = mf.Tools(workspace)

op = tools.validate(mf.ToolCall("c1", "read_file", '{"path": "calc.py"}'))
print("Operation:", op)
print()
print("display（给人看）:", op.display)
print("target （policy 拿来匹配规则）:", op.target)

`Operation` 被 policy 判、被审批框显示、被执行器执行——**三者拿到同一个对象**。

如果显示的和执行的可能不同，那用户批准的就不是实际发生的事，审批就成了摆设。

In [ ]:
session = fresh_session("tools.jsonl")
print(tools.run(op, session))

In [ ]:
# 边界检查：所有文件工具的地基
for path in ["calc.py", "../../../etc/passwd", "sub/../calc.py"]:
    try:
        resolved = tools._resolve(path)
        print(f"{path:26} -> OK   {resolved.name}")
    except ValueError as exc:
        print(f"{path:26} -> 拒绝 {exc}")

demo 只做了**前缀比较**。真 Foundry 的 `core/workspace.py` 还要处理 8.3 短名、
reparse point（Windows junction 不是 symlink，`islink` 返回 False）、大小写、
设备名（`CON`/`NUL`）、盘符相对路径（`C:foo`）、UNC。

这不是过度设计——上面每一样都是真的能绕过前缀比较的写法。

---
## 3. Policy —— 哪些能做、哪些要问、哪些永远不行

对应 `core/policy/`。**顺序就是全部含义**：

| 步 | 做什么 |
|---|---|
| **0** | **熔断表 —— 任何规则、模式、授权、hook 都无法覆盖** |
| 1 | DENY 规则 |
| 2 | ASK 规则 |
| 3 | 模式基线（只读 / 自动改 / 全问 / 全拒） |
| 4 | ALLOW 规则 |
| 5 | 问人 |

下面把一堆操作丢进去，看判决表。

In [ ]:
policy = mf.Policy(mode="default", allow_rules=["python -m pytest"])

samples = [
    ("read_file",   {"path": "calc.py"}),
    ("run_command", {"command": "python -m pytest -q"}),
    ("run_command", {"command": "python -m pytest -q --tb=short"}),
    ("run_command", {"command": "npm test"}),
    ("write_file",  {"path": "calc.py", "content": "x"}),
    ("run_command", {"command": "git status"}),
    ("run_command", {"command": "git push origin main"}),
    ("run_command", {"command": "git reset --hard HEAD"}),
    ("run_command", {"command": "rm -rf / --no-preserve-root"}),
]

print(f"{'操作':44} {'判决':6} 步  理由")
print("─" * 100)
for name, args in samples:
    op = tools.validate(mf.ToolCall("c", name, json.dumps(args)))
    d = policy.evaluate(op)
    print(f"{op.display[:44]:44} {d.verdict:6} {d.step}   {d.reason}")

注意两件事：

1. `python -m pytest -q --tb=short` 也被放行了——ALLOW 规则用的是前缀匹配。
2. 最后三条是**第 0 步**拒绝的。它们不是"优先级很高的规则"，而是在流水线之外。

下面验证第二点：给自己开一条最宽的 ALLOW 规则，看能不能放行 `git push`。

In [ ]:
greedy = mf.Policy(mode="accept_edits", allow_rules=[""])   # 空前缀 = 匹配一切
greedy.session_grants.add("git push origin main")           # 再加一条会话授权

op = tools.validate(mf.ToolCall("c", "run_command", '{"command": "git push origin main"}'))
d = greedy.evaluate(op)
print(f"最宽的 ALLOW 规则 + 会话授权 -> {d.verdict}（第 {d.step} 步）")
print(d.reason)

**这就是熔断表存在的理由。** 规则表是可配置的——仓库里的 `.foundry/config.toml`、
用户配置、hook 都能往里加东西。熔断表不能是其中一条，因为能被配置的东西就能被绕过。

真 Foundry 为此有一条结构性不变量测试：**540 个自动生成的组合**，断言任何装饰、
模式、会话授权或 hook 改写，都不能让一条被禁命令变得可批准。这条测试的由来是：
四轮对抗评审每一轮都攻破过命令分段器，其中两次是把「不可批准的 DENY」悄悄降级
成了「可批准的 ASK」。

In [ ]:
# 三种模式对同一批操作的差别
ops = [tools.validate(mf.ToolCall("c", n, json.dumps(a))) for n, a in samples[:6]]

print(f"{'操作':44} " + "".join(f"{m:14}" for m in ["default", "accept_edits", "plan"]))
print("─" * 92)
for op in ops:
    row = f"{op.display[:44]:44} "
    for mode in ["default", "accept_edits", "plan"]:
        row += f"{mf.Policy(mode=mode, allow_rules=['python -m pytest']).evaluate(op).verdict:14}"
    print(row)

---
## 4. Session —— 发生过什么的账本

对应 `core/session.py`。只追加，一行一个 JSON。两个用途：

1. 出了事能查——谁批准了什么、跑了什么命令、结果如何；
2. **证据链**——`finish` 声称"测试通过了"时，回来查那条命令的 exit code。

In [ ]:
workspace = fresh_workspace()
tools = mf.Tools(workspace)
session = fresh_session("evidence.jsonl")

op = tools.validate(mf.ToolCall("c", "run_command", '{"command": "python -m pytest -q"}'))
result = tools.run(op, session)
print(result[:200])
print()
print("账本里：")
for entry in session.events:
    print("  ", json.dumps(entry, ensure_ascii=False))

留意工具结果末尾的 `[event_id=N]`。**模型看得见它**——这就是它在 `finish` 里
引用证据的方式。下面手动核对一次。

In [ ]:
event_id = session.events[-1]["n"]
recorded = session.find_command(event_id)
print(f"事件 {event_id}:", recorded)
print()
print("这条命令能支持「测试通过了」吗？", recorded["exit_code"] == 0)

---
## 5. Backend —— 跟模型说话

对应 `core/backends/`。它只做**翻译**：IR 进去、厂商 JSON 出来；厂商 JSON 进来、
IR 出去。它绝对不能自己跑循环——否则就有两个地方在决定"下一步做什么"了。

### 切换模式

把 `USE_API` 改成 `True` 就用真模型。`ENDPOINT` 需要一个 **OpenAI 兼容的
`/v1/chat/completions`**，下面几种都行：

| 来源 | ENDPOINT | API_KEY |
|---|---|---|
| 本机 OpenClaw 网关 | `http://127.0.0.1:18789/v1` | `~/.openclaw/openclaw.json` 里的 `gateway.auth.token` |
| LM Studio | `http://127.0.0.1:1234/v1` | 随便填 |
| Ollama | `http://127.0.0.1:11434/v1` | 随便填 |
| OpenAI | `https://api.openai.com/v1` | 你的 key |

> **两个要先知道的事**
>
> 1. 本机 OpenClaw 网关背后的模型是 trade-advisor 系的，**它们不产出 tool_calls**。
>    所以 API 模式下 loop 会在第一轮就结束——你能看到真实的 HTTP 往返和真实的回答，
>    但看不到多轮工具调用。想看完整 loop，用剧本模式，或者换一个会调工具的模型
>    （LM Studio 里随便一个支持 function calling 的都行）。
> 2. **真 agent 一轮可能要几十秒到几分钟。** 下面把超时设成 120 秒；第 6 节那个
>    完整 loop 在 API 模式下会更慢，而且每一轮都花你的额度。

In [ ]:
USE_API  = False                                   # ← 改成 True 用真模型
ENDPOINT = "http://127.0.0.1:18789/v1"
MODEL    = "openclaw/trade-advisor-panel"
API_KEY  = "any-value"
TIMEOUT  = 120                                     # 秒。真 agent 一轮可能几分钟

# 本机 OpenClaw 网关的 token 就在配置文件里，顺手读出来（不打印）
import os
_cfg = Path(os.path.expanduser("~")) / ".openclaw" / "openclaw.json"
if USE_API and API_KEY == "any-value" and _cfg.is_file():
    try:
        API_KEY = json.loads(_cfg.read_text("utf-8"))["gateway"]["auth"]["token"]
        print("已从 ~/.openclaw/openclaw.json 读到 token（未打印）")
    except (KeyError, ValueError):
        pass


def make_backend(script="fix"):
    if USE_API:
        print(f"API 模式：{MODEL} @ {ENDPOINT}")
        return mf.HttpBackend(ENDPOINT, MODEL, API_KEY, timeout=TIMEOUT)
    print(f"剧本模式：{script}（不联网）")
    return mf.ScriptedBackend(mf.SCRIPTS[script])


backend = make_backend()

In [ ]:
# 先问最轻的一句，确认这条线是通的（真 agent 想事情可能要几十秒）
probe = "只回复两个字符：OK" if USE_API else "calc.py 里的 add 有 bug，修好它。"
reply = backend.sample([mf.Message("user", probe)], mf.Tools.schemas())
print("说了什么 :", reply.text or "(没说话)")
print("要调工具 :", [(c.name, c.arguments) for c in reply.tool_calls] or "(没有)")

if USE_API and not reply.tool_calls:
    print()
    print("↑ 这个模型不调工具，所以 loop 会在第一轮结束。见上面那段说明。")

---
## 6. Loop —— 把上面五个缝起来

对应 `core/runtime.py`。这就是整个系统，三十来行。

真的那个多了预算上限、取消、凭证过期重取、错误分类学、上下文窗口管理——
但**形状一模一样**。

In [ ]:
# 这一格**固定用剧本**，即使 USE_API=True。
# 理由上一节说过：完整 loop 的看点是多轮工具调用，而本机网关背后的模型不产出
# tool_calls，用它跑这格只会在第一轮就结束（而且慢）。想用 API 跑，见下一格。
workspace = fresh_workspace()
session   = fresh_session("full.jsonl")

code_ = mf.run(
    task="calc.py 里的 add 有 bug，修好它并确认测试通过。",
    backend=mf.ScriptedBackend(mf.SCRIPTS["fix"]),
    tools=mf.Tools(workspace),
    policy=mf.Policy(mode="default", allow_rules=["python -m pytest"]),
    session=session,
    auto="y",                     # 所有审批自动同意；改成 None 会真的问你
)
session.close()
print()
print("退出码:", code_, " (0=完成 10=部分完成)")
print("最终的 calc.py:", (workspace / "calc.py").read_text(encoding="utf-8").strip())

### 想用真模型跑整个 loop

下面这格默认不跑（`RUN_LIVE_LOOP = False`）。打开它之前先知道：

- 每一轮都是一次真实请求，**花你的额度**；
- 本机网关的模型不调工具，所以它大概率一轮就结束；
- 一轮可能几十秒到几分钟，超时会抛 `RuntimeError` 并说明原因。

想看到真正的多轮，换一个支持 function calling 的模型（LM Studio 里随便一个都行），
把上面的 `ENDPOINT` / `MODEL` 指过去。

In [ ]:
RUN_LIVE_LOOP = False            # ← 改成 True

if RUN_LIVE_LOOP and USE_API:
    workspace = fresh_workspace()
    session   = fresh_session("live.jsonl")
    try:
        mf.run(task="calc.py 里的 add 有 bug，修好它并确认测试通过。",
               backend=make_backend(),
               tools=mf.Tools(workspace),
               policy=mf.Policy(mode="default", allow_rules=["python -m pytest"]),
               session=session, auto="y")
    except RuntimeError as exc:
        print("后端出错：", exc)
    finally:
        session.close()
elif RUN_LIVE_LOOP:
    print("USE_API 还是 False——先到第 5 节把它打开。")
else:
    print("没跑。把 RUN_LIVE_LOOP 改成 True 再执行这一格。")

In [ ]:
# 完整账本
for entry in session.events:
    payload = json.dumps(entry["payload"], ensure_ascii=False)
    print(f"{entry['n']:>3}  {entry['type']:18} {payload[:88]}")

---
## 7. 三件被拦住的事

下面三格都用剧本模式（要看确定的行为），和 `USE_API` 无关。

### (a) 熔断表：`--yes` 也批不动

In [ ]:
session = fresh_session("destructive.jsonl")
mf.run(task="清理一下工作区。",
       backend=mf.ScriptedBackend(mf.SCRIPTS["destructive"]),
       tools=mf.Tools(fresh_workspace()),
       policy=mf.Policy(mode="default"),
       session=session, auto="y")          # ← 全部自动同意，仍然拦住
session.close()

### (b) 收工闸门：假的"测试全过"

模型**没有**伪造引用——它引用的命令真的跑过，只是失败了。拦住它的不是
「检测撒谎」，是**要求它指出证据，然后我们自己去查那条证据**。

In [ ]:
session = fresh_session("liar.jsonl")
mf.run(task="确认测试通过。",
       backend=mf.ScriptedBackend(mf.SCRIPTS["liar"]),
       tools=mf.Tools(fresh_workspace()),
       policy=mf.Policy(mode="default", allow_rules=["python -m pytest"]),
       session=session, auto="y")
session.close()

### (c) plan 模式：不做任何改动

In [ ]:
session = fresh_session("plan.jsonl")
workspace = fresh_workspace()
mf.run(task="修好 add。",
       backend=mf.ScriptedBackend(mf.SCRIPTS["fix"]),
       tools=mf.Tools(workspace),
       policy=mf.Policy(mode="plan"),
       session=session, auto="y")
session.close()
print()
print("calc.py 没被动过:", (workspace / "calc.py").read_text(encoding="utf-8").strip())

---
## 8. 自己动手

下面几格是留给你改的。

### 加一条自己的熔断规则

In [ ]:
saved = list(mf.FORBIDDEN)
mf.FORBIDDEN.append(("npm publish", "永不发布 npm 包"))

op = tools.validate(mf.ToolCall("c", "run_command", '{"command": "npm publish --access public"}'))
print(mf.Policy().evaluate(op))

mf.FORBIDDEN[:] = saved          # 改回去，免得影响后面的格子

### 试试子串匹配挡不住什么

demo 的熔断表用的是 `pattern in target`——最朴素的子串匹配。真实世界里同一条
命令有很多写法。跑一下，看哪些溜过去了。

In [ ]:
evasions = [
    "git reset --hard HEAD",          # 老实写法
    "git reset ,--hard HEAD",         # PowerShell 数组逗号：git 收到的还是 --hard
    "git    reset   --hard",          # 多空格
    "git reset <# 注释 #> --hard",    # PowerShell 块注释
    "git.exe reset --hard",           # 带扩展名
    "&'git' reset --hard",            # 调用操作符
    "echo hi; git reset --hard",      # 藏在链式命令的第二段
]
p = mf.Policy()
for command in evasions:
    op = tools.validate(mf.ToolCall("c", "run_command", json.dumps({"command": command})))
    d = p.evaluate(op)
    mark = "拦住" if d.verdict == mf.DENY else "★溜过去了"
    print(f"{mark:10} {command}")

**这就是那 443 行分段器（`core/policy/segmenter.py`）存在的全部理由。**

真 Foundry 对每条命令做**两种独立读法**：自己的词法分析，加一个故意很笨的、
无视引号和注释的朴素切分。任一读法命中就拒绝——朴素读法骗不了，因为它不做
任何可以被欺骗的假设。

诚实的表述是：**纵深防御，不是保证**。第五轮评审证伪了"不再依赖词法"的说法。
细节见 [../docs/threat-model.md](../docs/threat-model.md)。

---
## demo **没有**的东西

砍掉是为了让骨架看得清。这些正是真 Foundry 花力气的地方：

- 流式输出（还要处理凭证跨两个分片的脱敏——真实跑起来时抓到过）
- 路径安全（8.3 短名、reparse point、设备名、UNC、大小写）
- 命令分段（上面刚看过子串匹配的下场）
- 上下文窗口管理、预算上限、取消、凭证过期重取
- 错误分类学（哪些重试、哪些停、`Retry-After` 怎么读）
- 补丁的锚定匹配与逐文件原子性（demo 直接整文件覆盖写）
- 证据的**顺序**：引用一条跑在改动**之前**的绿灯命令，同样要被降级

## 接着读

| 想看什么 | 去哪 |
|---|---|
| 每一节对应真实模块 | [README.md](README.md) |
| 保护什么、不保护什么 | [../docs/threat-model.md](../docs/threat-model.md) |
| 为什么这样设计 | [../docs/decision-log.md](../docs/decision-log.md) |
| 唯一的 loop | `../src/foundry/core/runtime.py` |